# Ecobici — EDA

Análisis exploratorio sobre los datos limpios de `data_cleaning.ipynb` (hay que correrlo antes para generar `data/clean/trips_12m.parquet`).

**Qué se reutiliza de `data_cleaning.ipynb`** (no se repite):
- La limpieza completa (fechas, duplicados, estaciones inexistentes, duración ≤ 0 o > 180 min, viajes de < 3 min en la misma estación, velocidad > 25 km/h, ventana de 12 meses).
- Las columnas auxiliares: `mes`, `fecha`, `hora_retiro`, `hora_arribo`, `dia_semana`, `es_habil`, `en_horario`, `retiro_subred`, `arribo_subred`, `grupo_retiro`, `grupo_arribo`, `duracion_min`, `distancia_km`, `velocidad_kmh`.
- El resumen por mes (`outputs/tablas/resumen_meses.csv`).

**Contenido**
1. Configuración y carga
2. Comparación entre meses
3. Distribución de llegadas (retiros) y devoluciones (arribos)
4. Estaciones con más demanda
5. Horas con más demanda
6. Estandarización
7. Tiempo de uso (uso válido)
8. Datos atípicos
9. ¿Se puede normalizar?
10. a 18. **Parte B** (contexto y operación): justificación de la subred, días atípicos de septiembre, estabilidad y factores de escenario, movimientos de camión, matriz origen-destino, limpieza y descriptivas por estación y hora, estaciones sistemáticamente desbalanceadas y conclusiones

Convención: **retiro** = un usuario llega a la estación y saca una bici; **arribo** = devolución de una bici.

## 1. Configuración y carga

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import Markdown, display

def explicar(texto):
    # Interpretación sencilla debajo de cada gráfica, en Markdown
    display(Markdown(texto))

pd.set_option("display.max_columns", None)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

RUTA_12M = next((p for p in (Path("data/clean/trips_13m.parquet"), Path("data/clean/trips_12m.parquet")) if p.exists()), Path("data/clean/trips_12m.parquet"))   # usa el que exista
RUTA_ESTACIONES = Path("data/raw/station_information.json")
DIR_FIG = Path("outputs/figuras")
DIR_TABLAS = Path("outputs/tablas")
DIR_FIG.mkdir(parents=True, exist_ok=True)
DIR_TABLAS.mkdir(parents=True, exist_ok=True)

MES_PRINCIPAL = "2026-09"
COL_RETIRO, COL_ARRIBO = "Ciclo_Estacion_Retiro", "Ciclo_EstacionArribo"
SUBRED = {"014": "Reforma", "027": "Reforma", "018": "Reforma", "271-272": "Buenavista", "273-274": "Buenavista"}
COLORES = {"Buenavista": "#d1495b", "Reforma": "#2e86ab"}
def tomar(x, n, seed=0):
    return x.sample(min(n, len(x)), random_state=seed)

DIAS = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]

cols = ["mes", "fecha", "retiro", "arribo", "hora_retiro", "hora_arribo", "dia_semana", "es_habil", "en_horario",
        COL_RETIRO, COL_ARRIBO, "grupo_retiro", "grupo_arribo", "retiro_subred", "arribo_subred",
        "duracion_min", "distancia_km", "velocidad_kmh"]
df = pd.read_parquet(RUTA_12M, columns=cols)

estaciones = pd.DataFrame(json.loads(RUTA_ESTACIONES.read_text())["data"]["stations"]).set_index("short_name")
nombre = estaciones["name"]

obj = df[df.mes == MES_PRINCIPAL]
print(f"12 meses: {len(df):,} viajes   |   {MES_PRINCIPAL}: {len(obj):,} viajes")

## 2. Comparación entre meses

Los meses tienen distinto número de días, así que además del total se compara **por día hábil** (lo que usa el modelo). La tabla `resumen_meses` ya viene de la limpieza; aquí se amplía con viajes por día y duración.

In [ ]:
resumen = pd.read_csv(DIR_TABLAS / "resumen_meses.csv", index_col="mes")

hab = df[df.es_habil]
por_dia = hab.groupby(["mes", "fecha"], observed=True).size()             # viajes (toda la ciudad) por día hábil
resumen["viajes_dia_habil_ciudad"] = por_dia.groupby("mes").mean().round(0)
resumen["viajes_dia_habil_std"]    = por_dia.groupby("mes").std().round(0)
resumen["dur_mediana_min"] = df.groupby("mes", observed=True)["duracion_min"].median().round(2)
resumen["dur_media_min"]   = df.groupby("mes", observed=True)["duracion_min"].mean().round(2)
resumen["pct_fin_de_semana"] = (100 * (~df.es_habil).groupby(df.mes, observed=True).mean()).round(1)
resumen.to_csv(DIR_TABLAS / "eda_comparacion_meses.csv")
resumen

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
meses = resumen.index.astype(str)

ax[0].bar(meses, resumen.viajes_limpios / 1e6, color="#7a8b99")
ax[0].set_title("Viajes totales por mes (millones)")

ax[1].errorbar(meses, resumen.viajes_dia_habil_ciudad / 1e3, yerr=resumen.viajes_dia_habil_std / 1e3, fmt="o-", capsize=3, color="#2e86ab")
ax[1].set_title("Viajes por día hábil, toda la ciudad (miles, ± 1 desv. est.)")

ax[2].plot(meses, resumen.retiros_subred_dia, "o-", color="#2e86ab", label="Retiros subred / día hábil")
ax[2].plot(meses, resumen.arribos_subred_dia, "s-", color="#d1495b", label="Arribos subred / día hábil")
ax[2].set_title("Subred (5 estaciones): demanda por día hábil"); ax[2].legend()

for a in ax: a.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.savefig(DIR_FIG / "eda_comparacion_meses.png", dpi=200, bbox_inches="tight"); plt.show()
prom = resumen.viajes_dia_habil_ciudad.mean()
v_obj = resumen.loc[MES_PRINCIPAL, "viajes_dia_habil_ciudad"]
ra = resumen.loc[MES_PRINCIPAL, "arribos_subred_dia"] / resumen.loc[MES_PRINCIPAL, "retiros_subred_dia"]
if ra > 1.05: sentido = f"llegan {ra:.1f} veces más bicis de las que salen: se acumulan y alguien tiene que sacarlas"
elif ra < 0.95: sentido = f"salen {1 / ra:.1f} veces más bicis de las que llegan: se vacían y alguien tiene que reponerlas"
else: sentido = "entran y salen casi las mismas bicis"
rep = "es un mes representativo, ni excepcionalmente alto ni bajo" if abs(v_obj / prom - 1) < 0.10 else "se aleja del promedio, hay que tenerlo en cuenta"
explicar(f"En un día hábil la ciudad tiene entre {resumen.viajes_dia_habil_ciudad.min() / 1e3:.0f} mil y {resumen.viajes_dia_habil_ciudad.max() / 1e3:.0f} mil viajes: "
     f"el mes más tranquilo es {resumen.viajes_dia_habil_ciudad.idxmin()} y el más movido {resumen.viajes_dia_habil_ciudad.idxmax()}. "
     f"{MES_PRINCIPAL} ({v_obj / 1e3:.1f} mil) queda {abs(v_obj / prom - 1):.0%} {'por debajo' if v_obj < prom else 'por encima'} del promedio: {rep}. "
     f"En la subred (gráfica de la derecha), en {MES_PRINCIPAL} {sentido}.")

## 3. Distribución de llegadas (retiros) y devoluciones (arribos)

Para cada estación de la subred se cuenta cuántos retiros (llegadas de usuarios) y cuántos arribos (devoluciones) hay en cada **intervalo de 15 min** de los días hábiles de septiembre.

Se usa la ventana **07:00–10:00**, la hora pico de la mañana, porque ahí la tasa es aproximadamente constante. Si se mezclara todo el día, las horas de poco y mucho movimiento harían parecer los conteos más dispersos de lo que son.

Se comparan dos modelos:
- **Poisson:** media = varianza (llegadas independientes con tasa constante).
- **Binomial negativa:** varianza > media (sobredispersión, tasa que varía de un día a otro).

Se reporta el índice de dispersión (varianza / media) y el AIC de cada ajuste (menor es mejor).

In [ ]:
HORA_INI, HORA_FIN, MIN_BIN = 7, 10, 15
slots = range(HORA_INI * 60 // MIN_BIN, HORA_FIN * 60 // MIN_BIN)
dias_obj = np.sort(obj.loc[obj.es_habil, "fecha"].unique())

def conteos(d, col_tiempo, col_est, est):
    """Conteos por intervalo de 15 min (con ceros) de una estación, en días hábiles de d."""
    s = d[d[col_est] == est]
    s = s[s.es_habil]
    m = (s[col_tiempo].dt.hour * 60 + s[col_tiempo].dt.minute) // MIN_BIN
    fecha = s[col_tiempo].dt.normalize()
    s = s[(m >= slots.start) & (m < slots.stop)]
    g = s.groupby([fecha[s.index], m[s.index]]).size()
    idx = pd.MultiIndex.from_product([dias_obj, slots])
    return g.reindex(idx, fill_value=0).to_numpy()

def ajustar(x):
    mu, var = x.mean(), x.var(ddof=1)
    out = {"n": len(x), "media": mu, "varianza": var, "indice_disp": var / mu if mu > 0 else np.nan}
    out["aic_poisson"] = 2 * 1 - 2 * stats.poisson.logpmf(x, mu).sum()
    if var > mu:
        p = mu / var; r = mu * p / (1 - p)
        out["aic_nbinom"] = 2 * 2 - 2 * stats.nbinom.logpmf(x, r, p).sum()
        out["nb_r"], out["nb_p"] = r, p
    else:
        out["aic_nbinom"], out["nb_r"], out["nb_p"] = np.nan, np.nan, np.nan
    out["mejor"] = "Binomial negativa" if out["aic_nbinom"] < out["aic_poisson"] else "Poisson"
    return out

series = {}
filas = []
for est in SUBRED:
    for tipo, col_t, col_e in [("retiros", "retiro", COL_RETIRO), ("arribos", "arribo", COL_ARRIBO)]:
        x = conteos(obj, col_t, col_e, est)
        series[(est, tipo)] = x
        filas.append({"estacion": est, "grupo": SUBRED[est], "tipo": tipo, **ajustar(x)})
ajustes = pd.DataFrame(filas).round(3)
ajustes.to_csv(DIR_TABLAS / "eda_ajuste_distribuciones.csv", index=False)
ajustes[["estacion", "grupo", "tipo", "n", "media", "varianza", "indice_disp", "aic_poisson", "aic_nbinom", "mejor"]]

In [ ]:
# Se juntan las estaciones de cada grupo: 4 paneles en vez de 10 (el detalle por estación está en la tabla de arriba)
GRUPOS = ["Buenavista", "Reforma"]
series_g = {(g, tipo): sum(series[(e, tipo)] for e in SUBRED if SUBRED[e] == g) for g in GRUPOS for tipo in ["retiros", "arribos"]}

fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for j, g in enumerate(GRUPOS):
    for i, tipo in enumerate(["retiros", "arribos"]):
        x = series_g[(g, tipo)]; a = ax[i, j]; f = ajustar(x)
        k = np.arange(0, x.max() + 2)
        a.hist(x, bins=np.arange(-0.5, x.max() + 1.5), density=True, color=COLORES[g], alpha=0.55, edgecolor="white")
        a.plot(k, stats.poisson.pmf(k, f["media"]), "k-", lw=1.5, label="Al azar (Poisson)")
        if not np.isnan(f["nb_r"]):
            a.plot(k, stats.nbinom.pmf(k, f["nb_r"], f["nb_p"]), "--", color="darkorange", lw=1.5, label="Con rachas (bin. negativa)")
        a.set_title(f"{g} · {tipo} (media {f['media']:.1f} por 15 min)", fontsize=10)
        a.set_xlabel("Bicis por 15 min")
        if j == 0: a.set_ylabel("Frecuencia relativa")
        if i == 0 and j == 0: a.legend(fontsize=8)
fig.suptitle(f"Retiros y devoluciones cada 15 min, {HORA_INI}:00–{HORA_FIN}:00, días hábiles de {MES_PRINCIPAL}", y=1.01, fontsize=12)
plt.tight_layout(); plt.savefig(DIR_FIG / "eda_hist_llegadas_devoluciones.png", dpi=200, bbox_inches="tight"); plt.show()

lineas = []
for (g, tipo), x in series_g.items():
    d = ajustar(x)["indice_disp"]
    lineas.append(f"- **{g}, {tipo}:** " + ("casi al azar." if d < 1.3 else f"en rachas, con {d:.1f} veces la variación que habría al azar."))
explicar(f"Cada gráfica junta las estaciones de un grupo y cuenta cuántas bicis se retiran o se devuelven cada 15 minutos, entre las {HORA_INI} y las {HORA_FIN} h. "
         f"Si ocurrieran al azar, las barras seguirían la línea negra; si llegan en rachas (ratos con muchas y ratos con casi ninguna), se parecen más a la naranja.\n\n"
         + "\n".join(lineas) +
         "\n\nDonde hay rachas, el modelo no puede usar una tasa fija para toda la ventana: tiene que dejar que cambie con la hora. El detalle por estación está en la tabla de arriba.")

In [ ]:
# Prueba directa de Poisson: con n grande, (n-1)·var/media ~ chi² (n-1) si los datos son Poisson (cola superior = sobredispersión)
res = []
for (est, tipo), x in series.items():
    n = len(x); T = (n - 1) * x.var(ddof=1) / x.mean()
    res.append({"estacion": est, "tipo": tipo, "estadistico": round(T, 1), "gl": n - 1,
                "p_valor": stats.chi2.sf(T, n - 1)})
pd.DataFrame(res).assign(conclusion=lambda t: np.where(t.p_valor < 0.05, "Rechaza Poisson (sobredispersión)", "Compatible con Poisson"))

## 4. Estaciones con más demanda

Demanda = retiros + arribos. Se compara el top de las 12 meses con el de septiembre y se ubica dónde caen las 5 estaciones de la subred.

In [ ]:
def tabla_demanda(d):
    r = d[COL_RETIRO].value_counts(); a = d[COL_ARRIBO].value_counts()
    t = pd.DataFrame({"retiros": r, "arribos": a}).fillna(0).astype(int)
    t["demanda"] = t.retiros + t.arribos
    t["balance"] = t.arribos - t.retiros          # >0: se llena, <0: se vacía
    t["nombre"] = nombre.reindex(t.index)
    t["ranking"] = t.demanda.rank(ascending=False, method="min").astype(int)
    return t.sort_values("demanda", ascending=False)

dem_12m, dem_obj = tabla_demanda(df), tabla_demanda(obj)
dem_12m.head(15).to_csv(DIR_TABLAS / "eda_top_estaciones_12m.csv")
print("Top 15 estaciones, 12 meses"); display(dem_12m.head(15))
print(f"Subred (ranking entre {len(dem_obj)} estaciones, {MES_PRINCIPAL})"); display(dem_obj.loc[list(SUBRED)])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
top = dem_12m.head(15).iloc[::-1]
ax[0].barh(top.index, top.retiros / 1e3, label="Retiros", color="#2e86ab")
ax[0].barh(top.index, top.arribos / 1e3, left=top.retiros / 1e3, label="Arribos", color="#d1495b")
ax[0].set_title("Top 15 por demanda (miles de eventos, 12 meses)"); ax[0].legend()

ax[1].hist(dem_12m.demanda / 1e3, bins=50, color="#7a8b99")
for e, g in SUBRED.items():
    ax[1].axvline(dem_12m.loc[e, "demanda"] / 1e3, color=COLORES[g], lw=2)
ax[1].set_title("Distribución de la demanda por estación\n(líneas: estaciones de la subred)"); ax[1].set_xlabel("Miles de eventos")

plt.tight_layout(); plt.savefig(DIR_FIG / "eda_estaciones_demanda.png", dpi=200, bbox_inches="tight"); plt.show()
pos = ", ".join(f"{e}: #{int(dem_12m.loc[e, 'ranking'])}" for e in SUBRED)
explicar(f"A la izquierda, las 15 estaciones con más movimiento del año (retiros + devoluciones); la primera es {dem_12m.index[0]} con {dem_12m.demanda.iloc[0] / 1e3:.0f} mil. "
     f"A la derecha, la mayoría de las {len(dem_12m)} estaciones tiene bastante menos (la mediana es {dem_12m.demanda.median() / 1e3:.0f} mil) y unas pocas destacan mucho. "
     f"Las líneas marcan las 5 de la subred; su posición entre todas es: {pos}.")

## 5. Horas con más demanda

In [ ]:
n_hab = df.loc[df.es_habil, "fecha"].nunique()
n_fin = df.loc[~df.es_habil, "fecha"].nunique()
r_h = df[df.es_habil].groupby("hora_retiro").size() / n_hab
a_h = df[df.es_habil].groupby("hora_arribo").size() / n_hab
r_f = df[~df.es_habil].groupby("hora_retiro").size() / n_fin

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].plot(r_h.index, r_h / 1e3, "o-", color="#2e86ab", label="Retiros (hábil)")
ax[0].plot(a_h.index, a_h / 1e3, "s--", color="#d1495b", label="Arribos (hábil)")
ax[0].plot(r_f.index, r_f / 1e3, "^:", color="gray", label="Retiros (fin de semana/festivo)")
ax[0].set_title("Toda la ciudad: eventos por día (miles), promedio 12 meses"); ax[0].set_xlabel("Hora"); ax[0].legend()
ax[0].set_xticks(range(0, 24, 2))

# Mapa de calor día de la semana × hora
dias_por_sem = df.groupby("dia_semana")["fecha"].nunique()
mapa = (df.groupby(["dia_semana", "hora_retiro"]).size().unstack().reindex(columns=range(24), fill_value=0).div(dias_por_sem, axis=0)) / 1e3
im = ax[1].imshow(mapa.values, aspect="auto", cmap="YlOrRd")
ax[1].set_yticks(range(7)); ax[1].set_yticklabels(DIAS); ax[1].set_xticks(range(0, 24, 2)); ax[1].set_xticklabels(range(0, 24, 2))
ax[1].set_title("Retiros por hora (miles/día), por día de la semana"); ax[1].set_xlabel("Hora"); ax[1].grid(False)
plt.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.savefig(DIR_FIG / "eda_horas_demanda.png", dpi=200, bbox_inches="tight"); plt.show()
h_man, h_tar, h_fin = r_h.loc[5:11].idxmax(), r_h.loc[15:21].idxmax(), r_f.idxmax()
explicar(f"Entre semana hay dos picos: a las {h_man}:00 ({r_h[h_man]:,.0f} retiros en esa hora, en toda la ciudad) y a las {h_tar}:00 ({r_h[h_tar]:,.0f}), "
     f"que corresponden a ir y volver del trabajo o la escuela. En fin de semana no hay picos marcados: el máximo es a las {h_fin}:00 y es más bajo ({r_f[h_fin]:,.0f}). "
     f"En el mapa de calor, entre más oscuro el cuadro, más viajes en esa hora.")

top_h = r_h.nlargest(3)
print("Horas pico de retiros (día hábil):", ", ".join(f"{h}:00 ({v:,.0f}/día)" for h, v in top_h.items()))

## 6. Estandarización

Se estandariza (z = (x − media) / desv. est.) para poner en la misma escala meses con niveles distintos: cuántas desviaciones se aleja cada mes del promedio anual, en viajes de la ciudad y en retiros de la subred. (El perfil horario de cada estación se ve en la sección 16.)

In [ ]:
z_mes = ((resumen.viajes_dia_habil_ciudad - resumen.viajes_dia_habil_ciudad.mean()) / resumen.viajes_dia_habil_ciudad.std()).round(2)
z_sub = ((resumen.retiros_subred_dia - resumen.retiros_subred_dia.mean()) / resumen.retiros_subred_dia.std()).round(2)
pd.DataFrame({"z_ciudad_viajes_dia_habil": z_mes, "z_subred_retiros_dia_habil": z_sub})

## 7. Tiempo de uso (uso válido)

`data_cleaning.ipynb` ya definió el **uso válido** al eliminar: duración ≤ 0, duración > 180 min, viajes de < 3 min con la misma estación y velocidad > 25 km/h. Aquí se describe qué tan largos son los viajes que quedaron y se revisa que esa definición tenga sentido.

In [ ]:
dur = df["duracion_min"]
pct = dur.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2)
print(pct.to_string())

rangos = pd.cut(dur, [0, 3, 10, 20, 30, 60, 120, 180], right=True)
print("\nDistribución por rango de duración (min):")
display(rangos.value_counts(normalize=True, sort=False).mul(100).round(2).rename("% de viajes").to_frame())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.hist(dur[dur <= 60], bins=np.arange(0, 61, 1), density=True, color="#7a8b99", edgecolor="white")
shape, loc, scale = stats.lognorm.fit(tomar(dur, 200_000), floc=0)
xs = np.linspace(0.1, 60, 300)
ax.plot(xs, stats.lognorm.pdf(xs, shape, loc, scale), "k-", label=f"Log-normal (σ={shape:.2f}, mediana={scale:.1f})")
ax.set_title("Duración del viaje (≤ 60 min)"); ax.set_xlabel("min"); ax.legend()

plt.tight_layout(); plt.savefig(DIR_FIG / "eda_tiempo_uso.png", dpi=200, bbox_inches="tight"); plt.show()
explicar(f"La mitad de los viajes dura menos de {dur.median():.0f} minutos y 9 de cada 10 duran menos de {dur.quantile(.9):.0f}. "
     f"La curva negra describe bien la forma: muchos viajes cortos y pocos largos. {(dur <= 30).mean():.0%} de los viajes dura 30 minutos o menos. "
     f"Se considera uso válido el que quedó después de la limpieza: sin viajes de más de 180 minutos, sin duraciones imposibles y sin sacar y devolver la bici en la misma estación sin viajar.")

In [ ]:
# Duración en la subred (septiembre): ¿cuánto tiempo está una bici fuera? Importa para el modelo de redistribución.
t = obj[obj.retiro_subred | obj.arribo_subred].groupby("grupo_retiro", observed=True)["duracion_min"].describe(percentiles=[.5, .9, .99]).round(1)
t

## 8. Datos atípicos

Los atípicos *extremos* de un viaje (duración, velocidad) ya se eliminaron en la limpieza. Aquí se buscan los que **quedaron**:

1. Viajes con duración atípica según la regla de Tukey sobre `log(duración)` (la duración es muy asimétrica, en escala logarítmica es más razonable).
2. **Días** atípicos en demanda, con z robusto (mediana y MAD), que no se deja arrastrar por los propios atípicos.

In [ ]:
logd = np.log(dur)
q1, q3 = logd.quantile([.25, .75]); iqr = q3 - q1
lo, hi = np.exp(q1 - 1.5 * iqr), np.exp(q3 + 1.5 * iqr)
n_alto, n_bajo = (dur > hi).sum(), (dur < lo).sum()
print(f"Cerca de Tukey sobre log(duración): [{lo:.1f}, {hi:.1f}] min")
print(f"Viajes por encima: {n_alto:,} ({100*n_alto/len(df):.2f}%)   por debajo: {n_bajo:,} ({100*n_bajo/len(df):.2f}%)")
print("Nota: los largos son viajes de ocio o uso prolongado, no errores; ya están acotados a 180 min. Se conservan.")

In [ ]:
# Días atípicos: viajes por día en toda la ciudad
diario = df.groupby("fecha").size().rename("viajes").to_frame()
diario["dia"] = diario.index.day_name()
diario["tipo"] = np.where(df.groupby("fecha")["es_habil"].first(), "hábil", "fin de semana / festivo")

def z_robusto(s):
    med = s.median(); mad = (s - med).abs().median()
    return 0.6745 * (s - med) / mad

diario["z_robusto"] = diario.groupby("tipo")["viajes"].transform(z_robusto)      # se compara hábil con hábil
atip = diario[diario.z_robusto.abs() > 3.5].sort_values("z_robusto")
atip.to_csv(DIR_TABLAS / "eda_dias_atipicos.csv")
print(f"{len(atip)} días atípicos (|z robusto| > 3.5) de {len(diario)}")
display(atip)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for tipo, c in [("hábil", "#2e86ab"), ("fin de semana / festivo", "gray")]:
    s = diario[diario.tipo == tipo]; ax.scatter(s.index, s.viajes / 1e3, s=14, color=c, label=tipo)
ax.scatter(atip.index, atip.viajes / 1e3, s=60, facecolors="none", edgecolors="red", label="atípico")
ax.set_title("Viajes por día (miles), toda la ciudad"); ax.legend(); ax.tick_params(axis="x", rotation=45)
habil = diario[diario.tipo == "hábil"]
plt.tight_layout(); plt.savefig(DIR_FIG / "eda_atipicos.png", dpi=200, bbox_inches="tight"); plt.show()
fmt = lambda s: ", ".join(f"{d:%d/%m/%Y}" for d in s.index)
if len(atip):
    explicar(f"Cada punto es un día. Los círculos rojos son los {len(atip)} días (de {len(diario)}) que se alejan mucho de lo normal para su tipo de día. "
         f"Los más bajos: {fmt(atip.nsmallest(3, 'z_robusto'))}. Los más altos: {fmt(atip.nlargest(3, 'z_robusto'))}. "
         f"Antes de usarlos para estimar la demanda normal conviene revisar si coinciden con festivos, lluvia o eventos.")
else:
    explicar(f"Cada punto es un día. Ninguno de los {len(diario)} días se aleja mucho de lo normal para su tipo de día.")

**Qué hacer con los días atípicos:** revisar si coinciden con lluvia, huelgas, marchas o eventos. Si tienen causa conocida y no representan un día normal, se pueden excluir al estimar tasas para el modelo (septiembre ya excluye los días 14 y 15 por el cierre de Buenavista y el 16 por festivo).

## 9. ¿Se puede normalizar?

"Normalizar" aquí es: ¿se pueden transformar estas variables para que se parezcan a una distribución normal? Se revisan tres candidatas:

1. **Viajes por día hábil** (ciudad).
2. **Duración del viaje**, que es muy asimétrica; se prueba `log` y Box-Cox.
3. **Conteos por 15 min** (tipo Poisson), donde se prueba `√x`, que estabiliza la varianza.

Con millones de observaciones cualquier prueba formal de normalidad (Shapiro, D'Agostino) rechaza por diferencias minúsculas, así que se mira sobre todo la **asimetría, la curtosis y el gráfico Q-Q**.

In [ ]:
def resumen_norm(x, nombre):
    x = np.asarray(x, dtype=float)
    return {"variable": nombre, "n": len(x), "asimetria": stats.skew(x), "curtosis_exceso": stats.kurtosis(x)}

# 1. Viajes por día hábil
v = habil.viajes.to_numpy(float)
# 2. Duración (muestra)
d_s = tomar(dur, 200_000, 1).to_numpy(float)
bc, lam = stats.boxcox(d_s)
# 3. Conteos de 15 min de la subred (todas las series juntas)
c = np.concatenate(list(series.values())).astype(float)

filas = [resumen_norm(v, "viajes/día hábil"), resumen_norm(np.log(v), "log(viajes/día hábil)"),
         resumen_norm(d_s, "duración (min)"), resumen_norm(np.log(d_s), "log(duración)"), resumen_norm(bc, f"Box-Cox(duración), λ={lam:.2f}"),
         resumen_norm(c, "conteo 15 min"), resumen_norm(np.sqrt(c), "√conteo 15 min")]
tabla_norm = pd.DataFrame(filas).round(3)
# Pruebas formales sobre una submuestra de 5000 (límite práctico de Shapiro)
rng = np.random.default_rng(0)
def sw(x): x = np.asarray(x, float); return stats.shapiro(rng.choice(x, min(5000, len(x)), replace=False)).pvalue
tabla_norm["p_shapiro"] = [sw(v), sw(np.log(v)), sw(d_s), sw(np.log(d_s)), sw(bc), sw(c), sw(np.sqrt(c))]
tabla_norm.to_csv(DIR_TABLAS / "eda_normalidad.csv", index=False)
tabla_norm

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 8))
pares = [("Viajes/día hábil", v, "log(viajes/día hábil)", np.log(v)),
         ("Duración (min)", d_s, "log(duración)", np.log(d_s)),
         ("Conteo 15 min", c, "√conteo 15 min", np.sqrt(c))]
for j, (n1, x1, n2, x2) in enumerate(pares):
    stats.probplot(x1, dist="norm", plot=ax[0, j]); ax[0, j].set_title(f"Q-Q: {n1}")
    stats.probplot(x2, dist="norm", plot=ax[1, j]); ax[1, j].set_title(f"Q-Q tras transformar: {n2}")
plt.tight_layout(); plt.savefig(DIR_FIG / "eda_normalizacion_qq.png", dpi=200, bbox_inches="tight"); plt.show()
t_ = tabla_norm.set_index("variable")["asimetria"]
mejora_v = abs(t_["log(viajes/día hábil)"]) < abs(t_["viajes/día hábil"])
queda_c = abs(t_["√conteo 15 min"]) > 0.5
explicar(f"Si los puntos siguen la línea recta, los datos tienen forma de campana (normal). Arriba están los datos originales y abajo ya transformados. "
     f"La duración del viaje es muy asimétrica ({t_['duración (min)']:.1f}), pero con logaritmo queda casi simétrica ({t_['log(duración)']:.1f}): ahí sí se puede normalizar. "
     f"Los viajes por día {'mejoran' if mejora_v else 'no mejoran'} con logaritmo ({t_['viajes/día hábil']:.1f} → {t_['log(viajes/día hábil)']:.1f}), "
     f"{'' if mejora_v else 'porque sus valores raros son días bajos, no altos: ahí ayuda más quitar los días festivos. '}"
     f"Los conteos cada 15 minutos con raíz cuadrada pasan de {t_['conteo 15 min']:.1f} a {t_['√conteo 15 min']:.1f}"
     f"{' y todavía no son una campana: conviene modelarlos como conteos.' if queda_c else ', casi simétricos.'}")

In [ ]:
# Relación media–varianza de los conteos (Poisson: var = media). Tras √x la varianza debería ser ≈ 0.25 constante.
mv = pd.DataFrame({"media": [x.mean() for x in series.values()], "var": [x.var(ddof=1) for x in series.values()],
                   "var_sqrt": [np.sqrt(x).var(ddof=1) for x in series.values()]}, index=[f"{e} {t}" for e, t in series]).round(2)
mv

# Parte B — Contexto y operación de la subred

Las secciones 10 a 15 siguen la especificación `01B_contexto`: justifican la subred y el mes, dan los factores de escenario, reconstruyen la política actual de Ecobici (camión) y miden qué tan conectadas están las estaciones.

**Definiciones (se usan siempre las mismas):**
- *Días hábiles:* valores únicos de `fecha` con `es_habil`.
- *Retiros:* `es_habil & en_horario & retiro_subred`, con la fecha y hora del retiro.
- *Devoluciones:* `arribo_subred`, con la fecha y hora **del arribo**, en días hábiles según la fecha del arribo y hora entre 5 y 23.
- *Mañana* = 6:00 a 9:59; *tarde* = 16:00 a 19:59.
- *Saldo* = devoluciones − retiros.

## 10. Preparación de la subred

In [ ]:
ORDEN = ["271-272", "273-274", "014", "027", "018"]
FESTIVOS_SEP = pd.to_datetime(["2026-09-16"])
IRREGULARES = pd.to_datetime(["2026-09-14", "2026-09-15"])
MANANA, TARDE = (6, 9), (16, 19)                    # horas inclusivas
HORAS = list(range(5, 24))

habiles = pd.DatetimeIndex(np.sort(df.loc[df.es_habil, "fecha"].unique()))
meses = sorted(df.mes.astype(str).unique())
mes_de = lambda idx: pd.DatetimeIndex(idx).strftime("%Y-%m")
dias_por_mes = pd.Series(1, index=habiles).groupby(mes_de(habiles)).sum()
print("Meses en el archivo:", meses[0], "→", meses[-1])
print(f"Días hábiles en {MES_PRINCIPAL}: {dias_por_mes[MES_PRINCIPAL]}  (esperado: 19)")

# Retiros de la subred (fecha y hora del retiro)
ret = df.loc[df.es_habil & df.en_horario & df.retiro_subred, ["mes", "fecha", "hora_retiro", COL_RETIRO, COL_ARRIBO, "grupo_retiro", "grupo_arribo", "duracion_min"]].copy()
ret["mes"] = ret["mes"].astype(str)
ret = ret.rename(columns={"hora_retiro": "hora", COL_RETIRO: "est", COL_ARRIBO: "est_arribo", "grupo_retiro": "grupo"})
ret["est"] = ret["est"].astype(str); ret["est_arribo"] = ret["est_arribo"].astype(str); ret["grupo"] = ret["grupo"].astype(str)

# Devoluciones de la subred (fecha y hora del arribo)
dev = df.loc[df.arribo_subred, ["arribo", "hora_arribo", COL_ARRIBO, "grupo_arribo"]].copy()
dev["fecha"] = dev["arribo"].dt.normalize()
dev = dev[dev.fecha.isin(habiles) & dev.hora_arribo.between(5, 23)]
dev = dev.rename(columns={"hora_arribo": "hora", COL_ARRIBO: "est", "grupo_arribo": "grupo"})
dev["est"] = dev["est"].astype(str); dev["grupo"] = dev["grupo"].astype(str); dev["mes"] = mes_de(dev.fecha)
print(f"Retiros: {len(ret):,}   Devoluciones: {len(dev):,}")

## 11. Justificación de la subred

Tabla por estación en el mes objetivo, y su posición entre **todas** las estaciones del sistema por viajes y por desbalance de la mañana. Si están arriba en ambos rankings, la subred no es una muestra cualquiera.

La distancia al Metro o Metrobús más cercano necesita las estaciones de https://datos.cdmx.gob.mx (Secretaría de Movilidad). Descárgalas a `data/raw/transporte/` (CSV o GeoJSON); si la carpeta está vacía, esa columna se omite y el resto corre igual.

In [ ]:
def franja_mask(h, f): return h.between(f[0], f[1])

o_ret = ret[ret.mes == MES_PRINCIPAL]; o_dev = dev[dev.mes == MES_PRINCIPAL]
n_obj = int(dias_por_mes[MES_PRINCIPAL])

# --- Para todas las estaciones del sistema (ranking). Retiros de todas, no solo subred:
t = obj[obj.es_habil & obj.en_horario]
r_all = t.groupby(COL_RETIRO, observed=True).size()
r_man = t[franja_mask(t.hora_retiro, MANANA)].groupby(COL_RETIRO, observed=True).size()
r_tar = t[franja_mask(t.hora_retiro, TARDE)].groupby(COL_RETIRO, observed=True).size()
a = obj.assign(f_arr=obj.arribo.dt.normalize())
a = a[a.f_arr.isin(habiles) & a.hora_arribo.between(5, 23)]
d_all = a.groupby(COL_ARRIBO, observed=True).size()
d_man = a[franja_mask(a.hora_arribo, MANANA)].groupby(COL_ARRIBO, observed=True).size()
d_tar = a[franja_mask(a.hora_arribo, TARDE)].groupby(COL_ARRIBO, observed=True).size()

sis = pd.DataFrame({"retiros": r_all, "devoluciones": d_all, "r_man": r_man, "d_man": d_man, "r_tar": r_tar, "d_tar": d_tar}).fillna(0)
sis.index = sis.index.astype(str)
sis["viajes"] = sis.retiros + sis.devoluciones
sis["saldo_man"] = sis.d_man - sis.r_man
sis["saldo_tar"] = sis.d_tar - sis.r_tar
sis["rank_viajes"] = sis.viajes.rank(ascending=False, method="min").astype(int)
sis["rank_desbalance_man"] = sis.saldo_man.abs().rank(ascending=False, method="min").astype(int)

just = pd.DataFrame(index=ORDEN)
just["grupo"] = pd.Series(SUBRED)
just["nombre"] = nombre.reindex(ORDEN)
just["docks"] = estaciones["capacity"].reindex(ORDEN)
just["retiros_dia"] = (sis.retiros / n_obj).reindex(ORDEN).round(1)
just["devoluciones_dia"] = (sis.devoluciones / n_obj).reindex(ORDEN).round(1)
just["saldo_manana_dia"] = (sis.saldo_man / n_obj).reindex(ORDEN).round(1)
just["saldo_tarde_dia"] = (sis.saldo_tar / n_obj).reindex(ORDEN).round(1)
just["rank_viajes"] = sis.rank_viajes.reindex(ORDEN)
just["rank_desbalance_manana"] = sis.rank_desbalance_man.reindex(ORDEN)
just["n_estaciones_sistema"] = len(sis)

# --- Distancia al Metro / Metrobús (opcional)
def haversine_m(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    h = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * 6371000 * np.arcsin(np.sqrt(h))

def cargar_transporte(carpeta=Path("data/raw/transporte")):
    salida = []
    for f in sorted(carpeta.glob("*")) if carpeta.exists() else []:
        try:
            if f.suffix.lower() == ".csv":
                t = pd.read_csv(f, encoding_errors="replace")
                low = {c.lower(): c for c in t.columns}
                clat = next((low[k] for k in low if k in ("lat", "latitud", "stop_lat", "y", "geo_latitude")), None)
                clon = next((low[k] for k in low if k in ("lon", "lng", "long", "longitud", "stop_lon", "x", "geo_longitude")), None)
                cnom = next((low[k] for k in low if k in ("nombre", "name", "stop_name", "sitio", "estacion")), t.columns[0])
                if clat and clon:
                    s = pd.DataFrame({"nombre": t[cnom].astype(str), "lat": pd.to_numeric(t[clat], errors="coerce"), "lon": pd.to_numeric(t[clon], errors="coerce")})
                    salida.append(s.dropna().assign(fuente=f.name))
            elif f.suffix.lower() in (".geojson", ".json"):
                js = json.loads(f.read_text())
                for ft in js.get("features", []):
                    g = ft.get("geometry") or {}
                    if g.get("type") == "Point":
                        pr = ft.get("properties", {})
                        nm = next((str(v) for k, v in pr.items() if k.lower() in ("nombre", "name", "sitio", "estacion")), "s/n")
                        salida.append(pd.DataFrame([{"nombre": nm, "lat": g["coordinates"][1], "lon": g["coordinates"][0], "fuente": f.name}]))
        except Exception as e:
            print("No se pudo leer", f.name, "->", e)
    return pd.concat(salida, ignore_index=True) if salida else pd.DataFrame(columns=["nombre", "lat", "lon", "fuente"])

transporte = cargar_transporte()
if len(transporte):
    for e in ORDEN:
        d_m = haversine_m(estaciones.loc[e, "lat"], estaciones.loc[e, "lon"], transporte.lat.to_numpy(), transporte.lon.to_numpy())
        k = int(np.argmin(d_m)); just.loc[e, "dist_transporte_m"] = round(float(d_m[k]), 0); just.loc[e, "transporte_cercano"] = transporte.nombre.iloc[k]
    print(f"Transporte cargado: {len(transporte)} puntos de {sorted(transporte.fuente.unique())}")
else:
    print("AVISO: no hay archivos en data/raw/transporte/; se omite la distancia al Metro/Metrobús.")

just.to_csv(DIR_TABLAS / "justificacion_subred.csv")
just

In [ ]:
# Mapa con flujos reales de la mañana (grosor ∝ viajes por día) y % que sale de la red
def a_mercator(lat, lon):
    R = 6378137
    return R * np.radians(lon), R * np.log(np.tan(np.pi / 4 + np.radians(lat) / 2))

est5 = estaciones.loc[ORDEN, ["lat", "lon"]].copy()
est5["x"], est5["y"] = a_mercator(est5.lat, est5.lon)

def mapa_flujos(ax, franja, titulo):
    m = o_ret[franja_mask(o_ret.hora, franja)]
    flujo = m[m.est_arribo.isin(ORDEN)].groupby(["est", "est_arribo"]).size() / n_obj
    sale = 1 - (m[m.est_arribo.isin(ORDEN)].groupby("est").size() / m.groupby("est").size())
    for (i, j), v in flujo.items():
        if i == j: continue
        ax.annotate("", xy=(est5.loc[j, "x"], est5.loc[j, "y"]), xytext=(est5.loc[i, "x"], est5.loc[i, "y"]),
                    arrowprops=dict(arrowstyle="-|>", lw=0.4 + 0.5 * v, color="#555555", alpha=0.7, shrinkA=10, shrinkB=10, connectionstyle="arc3,rad=0.15"), zorder=2)
    for e in ORDEN:
        ax.scatter(est5.loc[e, "x"], est5.loc[e, "y"], s=200, color=COLORES[SUBRED[e]], edgecolor="white", lw=1.5, zorder=3)
        ax.annotate(f"{e}\nsale de la red: {100 * sale.get(e, np.nan):.0f}%", (est5.loc[e, "x"], est5.loc[e, "y"]), xytext=(10, 12),
                    textcoords="offset points", fontsize=8, weight="bold", bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.9), zorder=4)
    if len(transporte):
        tx, ty = a_mercator(transporte.lat.to_numpy(), transporte.lon.to_numpy())
        cerca = (tx > est5.x.min() - 800) & (tx < est5.x.max() + 800) & (ty > est5.y.min() - 800) & (ty < est5.y.max() + 800)
        ax.scatter(tx[cerca], ty[cerca], marker="^", s=60, color="black", zorder=3, label="Metro / Metrobús")
        ax.legend(loc="lower right")
    ax.set_xlim(est5.x.min() - 600, est5.x.max() + 900); ax.set_ylim(est5.y.min() - 500, est5.y.max() + 500)
    try:
        import contextily as ctx
        ctx.add_basemap(ax, crs="EPSG:3857", source=ctx.providers.Esri.WorldGrayCanvas)
    except Exception as e:
        print("Sin mapa de fondo:", type(e).__name__)
    ax.set_axis_off(); ax.set_title(titulo)

fig, ax = plt.subplots(1, 2, figsize=(16, 8))
mapa_flujos(ax[0], MANANA, "Mañana (6:00–9:59): viajes por día hábil")
mapa_flujos(ax[1], TARDE, "Tarde (16:00–19:59): viajes por día hábil")
plt.tight_layout(); plt.savefig(DIR_FIG / "mapa_subred_flujos.png", dpi=200, bbox_inches="tight"); plt.show()
m_man = o_ret[franja_mask(o_ret.hora, MANANA)]; m_tar = o_ret[franja_mask(o_ret.hora, TARDE)]
a_ref = (m_man[m_man.grupo == "Buenavista"].est_arribo.map(SUBRED) == "Reforma").mean()
t_bue = (m_tar[m_tar.grupo == "Reforma"].est_arribo.map(SUBRED) == "Buenavista").mean()
cierre_ = ("Es decir, la gente va y viene entre las mismas dos zonas, y por eso se pueden mover bicis dentro de la red."
           if min(a_ref, t_bue) > 0.5 else "Solo una parte de los viajes se queda dentro de la red.")
explicar(f"Cada flecha es un trayecto entre dos estaciones: entre más gruesa, más viajes por día. El porcentaje de cada estación es la parte de sus viajes que termina fuera de las cinco. "
     f"En la mañana, {a_ref:.0%} de los viajes que salen de Buenavista terminan en Reforma; en la tarde, {t_bue:.0%} de los que salen de Reforma terminan en Buenavista. {cierre_}")

## 12. Días atípicos de septiembre 2026

Serie diaria de viajes que tocan cada grupo de la subred. Se marcan fines de semana, el festivo del 16 y el cierre del 14–15 de Buenavista. Entre los hábiles que quedan, es atípico el día que se aleja **más de 25 %** de la mediana de su grupo.

*Si se propone excluir otro día, no se cambian los datos aquí: se reporta para acordarlo con el equipo, porque afecta también a la parte A.*

In [ ]:
dias_sep = pd.date_range(f"{MES_PRINCIPAL}-01", periods=30, freq="D")
def viajes_grupo(g):
    m = obj[(obj.grupo_retiro == g) | (obj.grupo_arribo == g)]
    return m.groupby("fecha").size().reindex(dias_sep, fill_value=0)

serie = pd.DataFrame({g: viajes_grupo(g) for g in ["Buenavista", "Reforma"]})
serie["tipo"] = np.select([serie.index.isin(IRREGULARES), serie.index.isin(FESTIVOS_SEP), serie.index.dayofweek >= 5],
                          ["cierre Buenavista", "festivo", "fin de semana"], "hábil")
base = serie[serie.tipo == "hábil"]
for g in ["Buenavista", "Reforma"]:
    med = base[g].median()
    serie[f"{g}_vs_mediana_%"] = ((serie[g] / med - 1) * 100).round(1)
    serie[f"{g}_atipico"] = (serie.tipo == "hábil") & ((serie[g] / med - 1).abs() > 0.25)

# Cierre del 14-15: caída frente a un día normal
normal = serie[serie.tipo == "hábil"]
for g in ["Buenavista", "Reforma"]:
    cierre = serie.loc[serie.tipo == "cierre Buenavista", g].mean(); med = normal[g].median()
    print(f"{g}: mediana de día hábil normal = {med:,.0f} viajes; días 14–15 = {cierre:,.0f} ({100 * (cierre / med - 1):+.0f}%)")
serie.to_csv(DIR_TABLAS / "eda_dias_septiembre.csv")
display(serie[(serie.tipo != "hábil") | serie.Buenavista_atipico | serie.Reforma_atipico])

**Qué se ve:** los números de caída del 14–15 se imprimen arriba. Para cada día de la tabla marcado como atípico entre los hábiles, se anota aquí si se mantiene o se propone excluir y por qué (por ejemplo, lluvia o un evento). Ningún día se excluye en este notebook.

## 13. Estabilidad entre meses y factores de escenario

Para la subred y los días hábiles, mes por mes:
- **Nivel:** retiros y devoluciones por día; se usa la **mediana** diaria (protege de días raros no marcados en otros meses) y también la media.
- **Forma:** perfil horario de retiros dividido entre el total del día; se mide la correlación con el perfil de septiembre 2026 y la máxima diferencia absoluta.
- **Patrón:** saldo de la mañana y la tarde por grupo.
- **Factores de escenario:** `f_m = mediana diaria de retiros del mes / la de septiembre 2026`.

*Limitación:* los días irregulares solo están marcados para septiembre 2026.

In [ ]:
def conteo_diario(ev, por):
    c = ev.groupby(["fecha", por]).size().unstack(fill_value=0).reindex(habiles, fill_value=0)
    return c

cd_ret = conteo_diario(ret, "grupo"); cd_dev = conteo_diario(dev, "grupo")
for c_ in (cd_ret, cd_dev):
    for g in ["Buenavista", "Reforma"]:
        if g not in c_.columns: c_[g] = 0
cd_ret["Subred"] = cd_ret[["Buenavista", "Reforma"]].sum(axis=1); cd_dev["Subred"] = cd_dev[["Buenavista", "Reforma"]].sum(axis=1)

mediana_ret = cd_ret.groupby(mes_de(cd_ret.index)).median()["Subred"]
factores = pd.DataFrame({"dias_habiles": dias_por_mes, "retiros_subred_mediana_dia": mediana_ret,
                         "retiros_subred_media_dia": cd_ret.groupby(mes_de(cd_ret.index)).mean()["Subred"],
                         "devoluciones_subred_mediana_dia": cd_dev.groupby(mes_de(cd_dev.index)).median()["Subred"]})
factores["factor_f_m"] = (factores.retiros_subred_mediana_dia / factores.loc[MES_PRINCIPAL, "retiros_subred_mediana_dia"]).round(3)
factores.to_csv(DIR_TABLAS / "factores_mes.csv")
factores

In [ ]:
# Forma del día: perfil horario de retiros / total del día
perf = ret.groupby(["mes", "grupo", "hora"]).size().unstack("hora", fill_value=0).reindex(columns=HORAS, fill_value=0)
perf = perf.div(perf.sum(axis=1), axis=0)

filas = []
fig, ax = plt.subplots(1, 2, figsize=(15, 4.3), sharey=True)
for a_, g in zip(ax, ["Buenavista", "Reforma"]):
    ref = perf.loc[(MES_PRINCIPAL, g)]
    for m in meses:
        if (m, g) not in perf.index: continue
        p = perf.loc[(m, g)]
        a_.plot(HORAS, p, lw=2.5 if m == MES_PRINCIPAL else 1, color="black" if m == MES_PRINCIPAL else None, alpha=1 if m == MES_PRINCIPAL else 0.6, label=m)
        filas.append({"mes": m, "grupo": g, "correlacion": np.corrcoef(p, ref)[0, 1], "max_dif_abs": (p - ref).abs().max()})
    a_.set_title(f"{g}: forma del día (retiros por hora / total)"); a_.set_xlabel("Hora"); a_.set_xticks(HORAS[::2])
ax[0].legend(ncol=3, fontsize=7)
plt.tight_layout(); plt.savefig(DIR_FIG / "eda_forma_dia_por_mes.png", dpi=200, bbox_inches="tight"); plt.show()
forma = pd.DataFrame(filas).round(3); forma.to_csv(DIR_TABLAS / "eda_forma_por_mes.csv", index=False)
explicar(f"Cada línea es un mes y muestra a qué horas ocurren los retiros, como fracción de los del día. Si las líneas casi se encimaran, el día tiene la misma forma todos los meses y solo cambia cuántos viajes hay. "
     f"El parecido con {MES_PRINCIPAL} va de {forma.correlacion.min():.2f} a {forma.correlacion.max():.2f} (1 es idéntico), así que "
     f"{'la forma del día es estable' if forma.correlacion.min() > 0.95 else 'hay meses con otra forma del día y conviene revisarlos'}.")
forma.pivot(index="mes", columns="grupo", values=["correlacion", "max_dif_abs"])

In [ ]:
# Patrón: saldo de la mañana y de la tarde por grupo y mes (promedio por día hábil)
def saldo_franja(f):
    r = ret[franja_mask(ret.hora, f)].groupby(["mes", "grupo"]).size()
    d = dev[franja_mask(dev.hora, f)].groupby(["mes", "grupo"]).size()
    s = (d.sub(r, fill_value=0)).unstack().reindex(meses)
    return s.div(dias_por_mes.reindex(meses), axis=0)

sal_man, sal_tar = saldo_franja(MANANA), saldo_franja(TARDE)
saldos = pd.concat({"manana": sal_man, "tarde": sal_tar}, axis=1).round(1)
saldos.to_csv(DIR_TABLAS / "eda_saldos_por_mes.csv")
display(saldos)
print("¿Buenavista < 0 y Reforma > 0 por la mañana en todos los meses?",
      bool((sal_man["Buenavista"] < 0).all() and (sal_man["Reforma"] > 0).all()))

In [ ]:
# Mismo mes del año anterior (septiembre 2025 vs septiembre 2026) y escenarios
ANT = f"{int(MES_PRINCIPAL[:4]) - 1}{MES_PRINCIPAL[4:]}"
if ANT in meses:
    comp = pd.DataFrame({ANT: [factores.loc[ANT, "retiros_subred_mediana_dia"], factores.loc[ANT, "devoluciones_subred_mediana_dia"],
                               sal_man.loc[ANT, "Buenavista"], sal_man.loc[ANT, "Reforma"], sal_tar.loc[ANT, "Buenavista"], sal_tar.loc[ANT, "Reforma"]],
                         MES_PRINCIPAL: [factores.loc[MES_PRINCIPAL, "retiros_subred_mediana_dia"], factores.loc[MES_PRINCIPAL, "devoluciones_subred_mediana_dia"],
                                         sal_man.loc[MES_PRINCIPAL, "Buenavista"], sal_man.loc[MES_PRINCIPAL, "Reforma"], sal_tar.loc[MES_PRINCIPAL, "Buenavista"], sal_tar.loc[MES_PRINCIPAL, "Reforma"]]},
                        index=["retiros/día (mediana)", "devoluciones/día (mediana)", "saldo mañana Buenavista", "saldo mañana Reforma", "saldo tarde Buenavista", "saldo tarde Reforma"])
    comp["dif_%"] = (100 * (comp[MES_PRINCIPAL] / comp[ANT] - 1)).round(1)
    comp.round(1).to_csv(DIR_TABLAS / "eda_sep_vs_sep.csv"); display(comp.round(1))
    print("Correlación de forma:", forma[(forma.mes == ANT)].set_index("grupo")["correlacion"].to_dict())
else:
    print(f"AVISO: {ANT} no está en el archivo (la ventana de data_cleaning.ipynb empieza en {meses[0]}). "
          "Para la comparación literal con el mismo mes del año anterior hay que ampliar la ventana a 13 meses (INICIO = '2025-09-01') y volver a correr la limpieza.")

esc = factores.factor_f_m
escenarios = pd.DataFrame({"escenario": ["bajo", "base", "alto"], "factor": [esc.min(), 1.0, esc.max()],
                           "mes": [esc.idxmin(), MES_PRINCIPAL, esc.idxmax()]})
escenarios.to_csv(DIR_TABLAS / "escenarios.csv", index=False)
print(f"Septiembre vs. promedio de la ventana: factor medio = {esc.mean():.3f}, mediana = {esc.median():.3f}")
escenarios

## 14. Movimientos de camión: la política actual de Ecobici

Los datos no registran al camión, pero el ID de la bici lo delata: si una bici **llega a A** y su **siguiente viaje sale de B ≠ A**, alguien la movió entre tanto.

1. Se ordena todo por `Bici` y `retiro`, usando el archivo grande para no cortar cadenas al cambio de mes.
2. Por bici: estación y hora del siguiente retiro.
3. Movimiento = estación de arribo ≠ estación del siguiente retiro.
4. Según el tiempo entre `arribo` y el siguiente retiro: hasta 24 h = **rebalanceo** (línea base); más de 24 h = **mantenimiento u otro** (no es redistribución operativa).
5. Se conservan los movimientos de septiembre 2026 en días hábiles que tocan la subred. La hora del movimiento se aproxima con la hora del siguiente retiro (es una cota superior del momento real).

*Es una reconstrucción, no un registro: la hora exacta no se conoce, y la limpieza (~0.2 % de viajes) puede cortar alguna cadena.*

In [ ]:
b = pd.read_parquet(RUTA_12M, columns=["Bici", COL_RETIRO, COL_ARRIBO, "retiro", "arribo"])
for c in (COL_RETIRO, COL_ARRIBO):
    b[c] = b[c].astype(str)
orden = np.lexsort((b["retiro"].to_numpy(), b["Bici"].cat.codes.to_numpy()))
b = b.iloc[orden].reset_index(drop=True)

misma = b["Bici"].cat.codes.to_numpy()[1:] == b["Bici"].cat.codes.to_numpy()[:-1]
origen = b[COL_ARRIBO].to_numpy()[:-1]          # donde quedó la bici
destino = b[COL_RETIRO].to_numpy()[1:]          # de donde salió después
t_llega = b["arribo"].to_numpy()[:-1]
t_sale = b["retiro"].to_numpy()[1:]
hueco_h = (t_sale - t_llega) / np.timedelta64(1, "h")

mov = misma & (origen != destino) & (hueco_h >= 0)
m = pd.DataFrame({"origen": origen[mov], "destino": destino[mov], "t_sale": pd.to_datetime(t_sale[mov]), "hueco_h": hueco_h[mov]})
m["tipo"] = np.where(m.hueco_h <= 24, "rebalanceo", "mantenimiento/otro")
del b
print(f"Movimientos detectados en 12 meses: {len(m):,}  ({(m.tipo == 'rebalanceo').mean():.1%} rebalanceo)")

m["fecha"] = m.t_sale.dt.normalize(); m["hora"] = m.t_sale.dt.hour
m["mes"] = mes_de(m.fecha)
sub = m[(m.mes == MES_PRINCIPAL) & m.fecha.isin(habiles) & (m.origen.isin(SUBRED) | m.destino.isin(SUBRED))]
print(f"Septiembre {MES_PRINCIPAL}, días hábiles, tocan la subred:")
display(sub.groupby("tipo").size().rename("movimientos").to_frame().assign(por_dia=lambda t: (t.movimientos / n_obj).round(1)))

In [ ]:
reb = sub[sub.tipo == "rebalanceo"]
sacadas = reb[reb.origen.isin(SUBRED)].groupby("origen").size().reindex(ORDEN, fill_value=0)
dejadas = reb[reb.destino.isin(SUBRED)].groupby("destino").size().reindex(ORDEN, fill_value=0)
cam_est = pd.DataFrame({"grupo": pd.Series(SUBRED), "sacadas_dia": sacadas / n_obj, "dejadas_dia": dejadas / n_obj}).loc[ORDEN]
cam_est["neto_dia"] = cam_est.dejadas_dia - cam_est.sacadas_dia
cam_est = cam_est.round(2); cam_est.to_csv(DIR_TABLAS / "camion_estacion.csv")
display(cam_est)

# Perfil por hora (hora del siguiente retiro = cota superior)
filas = []
for e in ORDEN:
    s = reb[reb.origen == e].groupby("hora").size().reindex(range(24), fill_value=0)
    d = reb[reb.destino == e].groupby("hora").size().reindex(range(24), fill_value=0)
    filas += [{"estacion": e, "hora": h, "sacadas_dia": s[h] / n_obj, "dejadas_dia": d[h] / n_obj} for h in range(24)]
cam_hora = pd.DataFrame(filas).round(3); cam_hora.to_csv(DIR_TABLAS / "camion_estacion_hora.csv", index=False)

# Flujos entre grupos
g_o = reb.origen.map(SUBRED).fillna("Fuera"); g_d = reb.destino.map(SUBRED).fillna("Fuera")
flujo_g = (pd.crosstab(g_o, g_d) / n_obj).round(2); flujo_g.index.name = "desde"; flujo_g.columns.name = "hacia"
print("Bicis movidas por el camión por día hábil, entre grupos:"); display(flujo_g)
print(f"Total de bicis movidas por día en la subred (rebalanceo): {len(reb) / n_obj:.1f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 4.3))
x = np.arange(5)
ax[0].bar(x - 0.2, cam_est.sacadas_dia, 0.4, label="Sacadas", color="#8a8a8a"); ax[0].bar(x + 0.2, cam_est.dejadas_dia, 0.4, label="Dejadas", color="#e0a030")
ax[0].set_xticks(x); ax[0].set_xticklabels(ORDEN); ax[0].set_title("Bicis movidas por el camión, por día hábil"); ax[0].legend()
for e in ORDEN:
    s = cam_hora[cam_hora.estacion == e]
    ax[1].plot(s.hora, s.sacadas_dia + s.dejadas_dia, "o-", ms=3, color=COLORES[SUBRED[e]], alpha=0.4 + 0.12 * ORDEN.index(e), label=e)
ax[1].set_title("Movimientos por hora (hora del siguiente retiro)"); ax[1].set_xlabel("Hora"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(DIR_FIG / "eda_camion.png", dpi=200, bbox_inches="tight"); plt.show()
tot_dia = len(reb) / n_obj
est_deja, est_saca = cam_est.dejadas_dia.idxmax(), cam_est.sacadas_dia.idxmax()
hora_pico = cam_hora.assign(t=cam_hora.sacadas_dia + cam_hora.dejadas_dia).groupby("hora").t.sum().idxmax()
explicar(f"No hay un registro del camión, pero se reconstruye siguiendo cada bici: si llega a una estación y su siguiente viaje sale de otra, alguien la movió. "
     f"En un día hábil, estos movimientos tocan la subred unas {tot_dia:.0f} veces. La estación donde más bicis se dejan es {est_deja} ({cam_est.loc[est_deja, 'dejadas_dia']:.1f} por día) "
     f"y de donde más se sacan, {est_saca} ({cam_est.loc[est_saca, 'sacadas_dia']:.1f} por día). La hora con más movimientos es las {hora_pico}:00, aproximada porque se usa la hora del siguiente retiro. "
     f"Esta es la política actual, contra la que se compararán las propuestas.")

## 15. Matriz origen-destino: qué tan conectadas están las estaciones

Para mañana, tarde y día completo (días hábiles de septiembre 2026): probabilidad de que un viaje que sale de la estación *i* termine en *j*, con una columna **Fuera** para los destinos fuera de la subred. También en sentido inverso (de dónde vienen las bicis que llegan).

In [ ]:
def destino_col(s): return s.where(s.isin(ORDEN), "Fuera")

FRANJAS = {"manana": MANANA, "tarde": TARDE, "dia": (5, 23)}

def od(franja, inversa=False):
    h = obj[obj.es_habil & obj.en_horario]
    if not inversa:
        d = h[h.retiro_subred & franja_mask(h.hora_retiro, FRANJAS[franja])]
        o, de = d[COL_RETIRO].astype(str), destino_col(d[COL_ARRIBO].astype(str))
        filas, cols = ORDEN, ORDEN + ["Fuera"]
    else:
        d = h[h.arribo_subred & franja_mask(h.hora_retiro, FRANJAS[franja])]
        o, de = destino_col(d[COL_RETIRO].astype(str)), d[COL_ARRIBO].astype(str)
        filas, cols = ORDEN + ["Fuera"], ORDEN
    tab = pd.crosstab(o, de).reindex(index=filas, columns=cols, fill_value=0)
    return tab.div(tab.sum(axis=1).replace(0, np.nan), axis=0), tab

resultados, cuentas = {}, {}
for f in FRANJAS:
    resultados[("sale_de", f)], cuentas[("sale_de", f)] = od(f)
    resultados[("llega_a", f)], cuentas[("llega_a", f)] = od(f, inversa=True)

od_largo = pd.concat({k: v for k, v in resultados.items()}, names=["sentido", "franja", "estacion"]).round(4)
od_largo.to_csv(DIR_TABLAS / "od_subred.csv")
print("Probabilidad de destino, viajes que salen de cada estación (día completo):"); display(resultados[("sale_de", "dia")].round(3))
print("Probabilidad de origen, viajes que llegan a cada estación (día completo):"); display(resultados[("llega_a", "dia")].round(3))

In [ ]:
# Versión agregada (Buenavista / Reforma / Fuera) y LaTeX escrito a mano
G = {**SUBRED}
def agregada(franja):
    t = cuentas[("sale_de", franja)].copy()
    t.index = t.index.map(G)
    t.columns = [G.get(c, "Fuera") for c in t.columns]
    t = t.groupby(level=0).sum().T.groupby(level=0).sum().T.reindex(index=["Buenavista", "Reforma"], columns=["Buenavista", "Reforma", "Fuera"], fill_value=0)
    return t.div(t.sum(axis=1), axis=0)

def a_latex(t, etiqueta):
    filas = ["\\begin{tabular}{l" + "r" * t.shape[1] + "}", "\\hline", " & " + " & ".join(t.columns) + " \\\\ \\hline"]
    for i, r in t.iterrows(): filas.append(f"{i} & " + " & ".join(f"{v:.3f}" for v in r) + " \\\\")
    filas += ["\\hline", "\\end{tabular}"]
    return f"% {etiqueta}\n" + "\n".join(filas)

latex = ""
for f in FRANJAS:
    ag = agregada(f); print(f"--- {f} ---"); display(ag.round(3)); latex += a_latex(ag, f"OD agregada, {f}") + "\n\n"
(DIR_TABLAS / "od_subred_agregada.tex").write_text(latex)

# Fracción que se queda dentro de la red, y tiempo de viaje entre grupos
dentro = pd.DataFrame({f: 1 - resultados[("sale_de", f)]["Fuera"] for f in FRANJAS}).loc[ORDEN]
tot = {f: 1 - cuentas[("sale_de", f)]["Fuera"].sum() / cuentas[("sale_de", f)].to_numpy().sum() for f in FRANJAS}
dentro.loc["TOTAL"] = pd.Series(tot)
print("Fracción de viajes que se queda dentro de la subred:"); display((dentro * 100).round(1))

tiempo = (o_ret[o_ret.est_arribo.isin(ORDEN)].assign(grupo_arribo=lambda t: t.est_arribo.map(SUBRED))
          .groupby(["grupo", "grupo_arribo"])["duracion_min"].agg(["median", "count"]).round(1))
print("Tiempo de viaje mediano entre grupos (min):"); display(tiempo)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a_, f in zip(ax, FRANJAS):
    ag = agregada(f)
    im = a_.imshow(ag.values, cmap="Blues", vmin=0, vmax=1)
    a_.set_xticks(range(3)); a_.set_xticklabels(ag.columns); a_.set_yticks(range(2)); a_.set_yticklabels(ag.index); a_.grid(False)
    for i in range(2):
        for j in range(3): a_.text(j, i, f"{ag.values[i, j]:.0%}", ha="center", va="center", color="white" if ag.values[i, j] > 0.5 else "black", fontsize=12)
    a_.set_title({"manana": "Mañana", "tarde": "Tarde", "dia": "Día completo"}[f])
fig.suptitle("¿A dónde van los viajes que salen de cada grupo?", y=1.03)
plt.tight_layout(); plt.savefig(DIR_FIG / "eda_od_agregada.png", dpi=200, bbox_inches="tight"); plt.show()
tot_dentro = dentro.loc["TOTAL", "dia"]
explicar(f"Cada cuadro responde: de 100 viajes que salen de ese grupo, ¿a dónde llegan? En la mañana, de cada 100 que salen de Buenavista, {100 * agregada('manana').loc['Buenavista', 'Reforma']:.0f} llegan a Reforma; "
     f"en la tarde, de cada 100 que salen de Reforma, {100 * agregada('tarde').loc['Reforma', 'Buenavista']:.0f} llegan a Buenavista. "
     f"Contando todo el día, {tot_dentro:.0%} de los viajes que salen de las cinco estaciones termina en otra de las cinco. "
     f"{'El resto sale de la subred, así que lo que pasa en una estación casi no determina lo que le llega a las otras.' if tot_dentro < 0.5 else 'Una parte importante se queda dentro, así que las estaciones sí se influyen entre sí.'}")

## 16. Limpieza documentada y estadística descriptiva por estación y hora

**Limpieza.** Las reglas y cuántos viajes quitó cada una vienen de `data_cleaning.ipynb` (`outputs/tablas/log_limpieza.csv`). Se muestran aquí para que el EDA quede autocontenido.

**Descriptivas.** Para las 5 estaciones de la subred en el mes objetivo: retiros, devoluciones y saldo por día hábil, y duración mediana, divididos en franjas del día.

In [ ]:
ruta_log = DIR_TABLAS / "log_limpieza.csv"
if ruta_log.exists():
    log = pd.read_csv(ruta_log)
    display(log)
    print(f"En total se eliminó el {100 * log.eliminados.sum() / log.restantes.iloc[0]:.2f}% de los viajes originales.")
else:
    print("AVISO: falta outputs/tablas/log_limpieza.csv; corre data_cleaning.ipynb.")

In [ ]:
FRANJAS_DIA = {"05:00-05:59": (5, 5), "Mañana 06:00-09:59": (6, 9), "Mediodía 10:00-15:59": (10, 15), "Tarde 16:00-19:59": (16, 19), "Noche 20:00-23:59": (20, 23)}
filas = []
for e in ORDEN:
    for nm, f in FRANJAS_DIA.items():
        r = o_ret[(o_ret.est == e) & franja_mask(o_ret.hora, f)]; d = o_dev[(o_dev.est == e) & franja_mask(o_dev.hora, f)]
        filas.append({"estacion": e, "grupo": SUBRED[e], "franja": nm, "retiros_dia": len(r) / n_obj, "devoluciones_dia": len(d) / n_obj,
                      "saldo_dia": (len(d) - len(r)) / n_obj, "duracion_mediana_min": r.duracion_min.median()})
desc = pd.DataFrame(filas).round(1); desc.to_csv(DIR_TABLAS / "estadisticas_estacion_franja.csv", index=False)
display(desc.pivot(index="estacion", columns="franja", values="saldo_dia").reindex(ORDEN))

In [ ]:
# Saldo medio por hora y estación: dónde y cuándo se llena o se vacía cada estación
h = (o_dev.groupby(["est", "hora"]).size().sub(o_ret.groupby(["est", "hora"]).size(), fill_value=0) / n_obj).unstack(fill_value=0)
h = h.reindex(index=ORDEN, columns=HORAS, fill_value=0)
lim = np.abs(h.values).max()
fig, ax = plt.subplots(figsize=(11, 3.3))
im = ax.imshow(h.values, aspect="auto", cmap="RdBu", vmin=-lim, vmax=lim)
ax.set_yticks(range(5)); ax.set_yticklabels([f"{e} ({SUBRED[e]})" for e in ORDEN]); ax.set_xticks(range(len(HORAS))); ax.set_xticklabels(HORAS); ax.grid(False)
ax.set_xlabel("Hora"); ax.set_title(f"Saldo medio por hora (devoluciones − retiros por día hábil), {MES_PRINCIPAL}")
plt.colorbar(im, ax=ax, label="bicis/día  (azul: se llena · rojo: se vacía)")
plt.savefig(DIR_FIG / "eda_saldo_hora_estacion.png", dpi=200, bbox_inches="tight"); plt.show()
s_h = h.stack(); (e_min, hr_min), (e_max, hr_max) = s_h.idxmin(), s_h.idxmax()
explicar(f"Azul: en esa hora la estación recibe más bicis de las que salen (se llena). Rojo: salen más de las que llegan (se vacía). "
     f"La que más se vacía es {e_min} ({SUBRED[e_min]}) a las {hr_min}:00, con unas {-s_h.min():.0f} bicis menos por día; la que más se llena es {e_max} ({SUBRED[e_max]}) a las {hr_max}:00, con {s_h.max():.0f} más por día.")

## 17. Estaciones sistemáticamente desbalanceadas

Una estación está **sistemáticamente desbalanceada** si casi todos los días hábiles termina la mañana con el mismo signo de saldo (siempre se llena o siempre se vacía) y la magnitud es relevante frente a su tamaño. Esto se evalúa en **todas** las estaciones del sistema, no solo en las 5.

Criterios (se pueden ajustar):
- `CONSISTENCIA ≥ 80 %`: fracción de días hábiles con el mismo signo que el saldo medio.
- `|saldo medio de la mañana| / docks ≥ 25 %`: el desbalance es grande para su capacidad.
- Prueba t de una muestra contra 0 con p < 0.01 (el saldo medio no es ruido).

In [ ]:
CONSISTENCIA, FRAC_DOCKS, ALFA = 0.80, 0.25, 0.01
hab_obj = pd.DatetimeIndex(np.sort(obj.loc[obj.es_habil, "fecha"].unique()))

t2 = t.assign(est=t[COL_RETIRO].astype(str))
a2 = a.assign(est=a[COL_ARRIBO].astype(str))
def saldo_dia_est(f):
    r = t2[franja_mask(t2.hora_retiro, f)].groupby(["est", "fecha"]).size()
    d = a2[franja_mask(a2.hora_arribo, f)].groupby(["est", "f_arr"]).size().rename_axis(["est", "fecha"])
    return d.sub(r, fill_value=0).unstack("fecha").reindex(columns=hab_obj).fillna(0)

sd_man, sd_tar = saldo_dia_est(MANANA), saldo_dia_est(TARDE)
est_ok = sd_man.index
rows = pd.DataFrame(index=est_ok)
rows["docks"] = estaciones["capacity"].reindex(est_ok).astype(float)
rows["saldo_man_medio"] = sd_man.mean(axis=1)
rows["saldo_tar_medio"] = sd_tar.reindex(est_ok).mean(axis=1)
rows["dias_consistentes"] = np.maximum((sd_man > 0).mean(axis=1), (sd_man < 0).mean(axis=1))
rows["p_valor"] = stats.ttest_1samp(sd_man.to_numpy(), 0, axis=1).pvalue
rows["frac_docks"] = rows.saldo_man_medio.abs() / rows.docks
rows["sistematica"] = (rows.dias_consistentes >= CONSISTENCIA) & (rows.frac_docks >= FRAC_DOCKS) & (rows.p_valor < ALFA)
rows["tipo_manana"] = np.where(~rows.sistematica, "equilibrada", np.where(rows.saldo_man_medio > 0, "se llena", "se vacía"))
rows["en_subred"] = rows.index.isin(ORDEN)
rows["nombre"] = nombre.reindex(est_ok)
rows = rows.sort_values("saldo_man_medio")
rows.round(3).to_csv(DIR_TABLAS / "desbalance_sistematico.csv")

n_sis = int(rows.sistematica.sum())
print(f"{n_sis} de {len(rows)} estaciones están sistemáticamente desbalanceadas por la mañana "
      f"({(rows.tipo_manana == 'se llena').sum()} se llenan, {(rows.tipo_manana == 'se vacía').sum()} se vacían).")
print("Estaciones de la subred:"); display(rows.loc[[e for e in ORDEN if e in rows.index], ["saldo_man_medio", "saldo_tar_medio", "dias_consistentes", "frac_docks", "tipo_manana"]].round(2))
print("Las 8 que más se vacían y las 8 que más se llenan:"); display(pd.concat([rows.head(8), rows.tail(8)])[["nombre", "docks", "saldo_man_medio", "saldo_tar_medio", "dias_consistentes", "tipo_manana"]].round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(rows.saldo_man_medio, rows.saldo_tar_medio, s=9, color="#c9c9c9", label="Equilibradas / no sistemáticas", zorder=1)
s_ = rows[rows.sistematica]
ax.scatter(s_.saldo_man_medio, s_.saldo_tar_medio, s=22, color="#555555", label="Sistemáticamente desbalanceadas", zorder=2)
for e in ORDEN:
    if e in rows.index:
        ax.scatter(rows.loc[e, "saldo_man_medio"], rows.loc[e, "saldo_tar_medio"], s=110, color=COLORES[SUBRED[e]], edgecolor="white", zorder=3)
        ax.annotate(e, (rows.loc[e, "saldo_man_medio"], rows.loc[e, "saldo_tar_medio"]), xytext=(6, 5), textcoords="offset points", fontsize=9)
ax.axhline(0, color="k", lw=0.7); ax.axvline(0, color="k", lw=0.7)
ax.set_xlabel("Saldo medio de la mañana (bicis/día)"); ax.set_ylabel("Saldo medio de la tarde (bicis/día)")
ax.set_title(f"Desbalance mañana vs tarde, {len(rows)} estaciones, {MES_PRINCIPAL}"); ax.legend(loc="upper right", fontsize=8)
plt.savefig(DIR_FIG / "eda_desbalance_sistematico.png", dpi=200, bbox_inches="tight"); plt.show()
inv = ((rows.saldo_man_medio * rows.saldo_tar_medio) < 0)[rows.sistematica].mean() if n_sis else float("nan")
k5 = int(rows.loc[[e for e in ORDEN if e in rows.index], "sistematica"].sum())
explicar(f"Cada punto es una estación: a la derecha gana bicis en promedio cada mañana y a la izquierda las pierde; arriba gana por la tarde y abajo pierde. "
     f"Una estación es “sistemática” si casi todos los días le pasa lo mismo y la diferencia es grande para su tamaño: {n_sis} de {len(rows)} lo son, y {k5} de las 5 de la subred (puntos de colores). "
     f"{inv:.0%} de las sistemáticas cambia de signo por la tarde (arriba a la izquierda o abajo a la derecha): la que se vacía en la mañana se llena en la tarde, y al revés.")

## 18. Conclusiones y preguntas del equipo

Se completa después de correr el notebook, con los números de las tablas:

1. **¿Por qué estas 5 estaciones?** `justificacion_subred.csv` y `desbalance_sistematico.csv` (sección 17).
2. **¿Por qué septiembre 2026? ¿Qué días se descartaron?** Sección 12 y `factores_mes.csv`.
3. **¿El patrón es estable entre meses y frente a septiembre 2025?** Sección 13 (forma, saldos, sep vs sep).
4. **Factores de escenario:** `escenarios.csv` (bajo, base, alto) y los meses que los definen.
5. **¿Cuántas bicis mueve hoy Ecobici en la subred, de dónde a dónde y a qué hora?** `camion_estacion*.csv`.
6. **¿Qué tan conectadas están las estaciones?** `od_subred.csv`.
7. **Limitaciones:** movimientos de camión reconstruidos; días irregulares marcados solo en septiembre 2026; capacidad tomada del feed actual; la ventana de limpieza no incluye septiembre 2025 salvo que se amplíe a 13 meses.

**Preguntas que va a responder el proyecto:**
- ¿Con cuántas bicis debe arrancar cada estación?
- ¿Cuándo y cuánto conviene mover con el camión?
- ¿Cuántos rechazos se evitan frente a lo que hace hoy Ecobici?
- ¿La recomendación aguanta en meses de demanda alta y baja?
- ¿Cuántos viajes en auto y emisiones se evitan?

**Salidas del EDA:** `outputs/figuras/eda_*.png`, `mapa_subred_flujos.png` y `outputs/tablas/` (`eda_*.csv`, `justificacion_subred.csv`, `factores_mes.csv`, `escenarios.csv`, `camion_estacion*.csv`, `od_subred*.csv/.tex`).